In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math
import datetime
from pyquaternion import Quaternion
import plotly.express as px
from ipywidgets import interact, interactive, fixed, interact_manual
import ipywidgets as widgets
import argparse

In [ ]:
report_df = pd.DataFrame(columns=['method','stage','user','score','headAngle_eu'])

In [ ]:
config_list = [
  'startedTime',
  'user',
  'stage',
  'beginScore',
  'mode',
  'fov',
  'method',
  'interaxial',
  'sensitivity',
  'frameLength',
]

result={
  'method': 'na',
  'stage': 'na',
  'user': 'na',
  'score': -1,
  'widePortion': -1,
  'transitionPortion': -1,
  'transitionCount': -1,
  'head_qu_distance': -1,
  'head_qu_mean_velocity': -1,
  'head_qu_mean_acc': -1,
  'rightHand_qu_distance': -1,
  'rightHand_qu_mean_velocity': -1,
  'rightHand_qu_mean_acc': -1,
  'leftHand_qu_distance': -1,
  'leftHand_qu_mean_velocity': -1,
  'leftHand_qu_mean_acc': -1,
  'player_mean_velocity': -1,
  'player_mean_acc': -1,
}

In [ ]:
userList = [
  'P002',
  'P004',
  'P009',
  'P011',
  'P012',
  'P013',
  'P026',
  'P027',
  'P028',
  'P029',
  'P032',
  'P033',
  'P035',
]

methodList = [
  'city_Dynamic_IdleDetection',
  'maze_Dynamic_IdleDetection',
  'city_Dynamic_TargetDetection',
  'shooting_Dynamic_TargetDetection',
  'puzzle_Dynamic_TargetDetection',
  'shooting_Dynamic_Manual',
  'maze_Dynamic_Manual',
  'puzzle_Dynamic_HandPosition',
  'city_Static',
  'puzzle_Static',
  'shooting_Static',
  'maze_Static'
]

In [ ]:
for userIdx in range(len(userList)):
  for methodIdx in range(len(methodList)):
    arg = "../data/chi24/formal/" + userList[userIdx] + "/LOG_Formal_" + userList[userIdx] + "_" + methodList[methodIdx] + "_0.txt"
    config = {}
    headAngle_eu = np.array([[0,0,0]])
    unstruct_tags = []
    unstruct_zoom_tags = []
    duration = 0.0

    completeFileName = arg

    f = open(completeFileName, 'r')
    count = 0
    frameCount = 0
    for i, line in enumerate(f):
      if i < len(config_list) and i == count:
        if (line.find("startedTime") == 0):
          startedTime = datetime.datetime.strptime(line[line.find(":")+1:].strip('\n'), '%m/%d/%Y %I:%M:%S %p')
      if i >= len(config_list) and i == count:
        if (line.find("endedTime") == 0):
          endedTime = datetime.datetime.strptime(line[line.find(":")+1:].strip('\n'), '%m/%d/%Y %I:%M:%S %p')
        if (line.find("*") != 0 and line.find("#") != 0):
          frameCount += 1
      count += 1

    print("startedTime: ", startedTime)
    print("endedTime: ", endedTime)
    actualDuration = (endedTime - startedTime).total_seconds()
    print("frameCount: ", frameCount)
    print("actual duration: ", actualDuration, "seconds")
    actualFrameRate = frameCount / actualDuration # in FPS
    print("actual frameRate: ", actualFrameRate, "FPS")
    turncatePoint = round(60 * actualFrameRate)
    print("trucate point frame: ", turncatePoint)

    f = open(completeFileName, 'r')
    count = 0
    frameCount = 0
    for i, line in enumerate(f):
      if (frameCount > turncatePoint):
        break
      if (line == "===\n"):
        break
      if i < len(config_list) and i == count:
        config[config_list[count]] = line[line.find(":")+1:].strip('\n')
      if i >= len(config_list) and i == count:
        if (line.find("*") == 0):
          unstruct_tags.append(str(frameCount) + line.split("\n")[0])
        elif (line.find("#") == 0):
          unstruct_zoom_tags.append(str(frameCount) + line.split("\n")[0])
        else:
          frameCount += 1
          
          headAngle_eu = np.append(headAngle_eu, [[ float(line.split("%")[1][1:].strip(')').split(",")[0]), 
                                                    float(line.split("%")[1][1:].strip(')').split(",")[1]), 
                                                    float(line.split("%")[1][1:].strip(')').split(",")[2]) ]], 
                                                    axis=0)

      count += 1

    # clear initialized zeros
    headAngle_eu = headAngle_eu[1:]
    duration = (1/actualFrameRate) * frameCount # trucated duration

    # check fov
    if (config['mode'] == 'Static'):
      config.update({'method': 'N/A'})
      config.update({'sensitivity': 'N/A'})
      if (config['fov'] != 'Normal'):
        print("\n⚠️ STATIC FOV IS NOT NORMAL")
    elif (config['mode'] == 'Dynamic'):
      config.update({'fov': 'N/A'})

    # check duration
    config.update({'frameLength': (1/actualFrameRate)})
    config.update({'duration': duration}) # trucated duration

    # check score
    if (config['stage'] == '3 maze'):
      config.update({'score': duration})
    else:
      config.update({'score': len(unstruct_tags)})
    if (config['beginScore'] != '0 Pt'):
      print("\n⚠️ BEGIN SCORE IS NOT 0")

    print("\nconfig", config)
    print("\nhead angle euler", headAngle_eu)

    report_df = pd.concat([report_df, pd.DataFrame([{ 'method': config['method'], 'stage': config['stage'], 'user': config['user'], 'score': config['score'], 'headAngle_eu': headAngle_eu }])], ignore_index=True)

In [ ]:
report_df[1:50]

In [ ]:
methods = ['N/A', 'Hand Position', 'Target Detection']
yaw_lists = []

for m, mm in enumerate(methods):
  tmp_list = []
  for i in range(report_df[(report_df['method'] == mm) & (report_df['stage'] == '1 puzzle')].shape[0]):
    tmp_list.append(report_df[(report_df['method'] == mm) & (report_df['stage'] == '1 puzzle')]['headAngle_eu'][:].values[i][:,1])

  # flat list
  tmp_list = [item for sublist in tmp_list for item in sublist]

  print(tmp_list)
  yaw_lists.append(tmp_list)

In [ ]:
N = 180

methods = ['None', 'Hand', 'Target']

for i in range(len(methods)):
  theta = np.linspace(0.0, 2 * np.pi, N, endpoint=False)
  radii = np.histogram(yaw_lists[i], bins=N, range=[0, 360])[0]
  width = (2*np.pi) / N

  ax = plt.subplot(polar=True)
  bars = ax.bar(theta, radii, width=width, bottom=max(np.histogram(yaw_lists[i], bins=N, range=[0, 360])[0])/3, color='slateblue')

  ax.set_title('Euler Head Yaw Direction Distribution in ' + methods[i] + ' method in puzzle stage')
  ax.set_xlabel('degree [every ' + str(360/N) + r'$\degree$' + ']')
  ax.set_ylabel('frequency', rotation=0, labelpad=-50)

  plt.show()

  # 0 to 90 degrees means heads down

In [ ]:
# turn into circular heatmap

N = 360  # Number of bins
methods = ['None', 'Hand', 'Target']

for i in range(len(yaw_lists)):
    theta = np.linspace(0.0, 2 * np.pi, N, endpoint=False)
    radii, _ = np.histogram(yaw_lists[i], bins=N, range=[0, 360])
    theta = np.append(theta, theta[0])  # To make it a closed loop
    radii = np.append(radii, radii[0])  # To make it a closed loop

    ax = plt.subplot(polar=True)
    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)

    # Create a 2D array of the radii to use pcolormesh
    radii_grid = np.array([radii, radii])
    theta_grid = np.array([theta, theta])

    # Plot the heatmap
    c = ax.pcolormesh(theta_grid, np.array([0, 1]), np.array(radii_grid), shading='auto', cmap='inferno')

    # ax.set_title('Euler Head Yaw Direction Distribution in ' + methods[i] + ' method in puzzle stage')
    ax.set_xlabel('')
    ax.set_ylabel('', rotation=0, labelpad=-50)

    ax.set_xticklabels([])
    ax.set_yticklabels([])
    ax.tick_params(axis='both', which='both', length=0)
    ax.grid(False)

    plt.colorbar(c, ax=ax, pad=0.1, label='Frequency')

    plt.show()
